# Exploratory Data Analysis

## Dataset Overview

The shape and structure of the dataset: rows, columns, types, missing values and what the values look like.

### O1. Setup and load

VS Code runs the notebook from `notebooks/`, so the data path is `../data/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

DATA = Path("../data/Flight_Data_With_Weather_Final.csv")
df = pd.read_csv(DATA, low_memory=False, parse_dates=["FL_DATE"])

### O2. Shape

In [ ]:
print(f"rows:    {df.shape[0]:,}")
print(f"columns: {df.shape[1]}")
print(f"memory:  {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB")
print(f"dates:   {df['FL_DATE'].min().date()} to {df['FL_DATE'].max().date()}")

### O3. Column summary

One row per column: its group, type, how much is missing, how many distinct values, and an example value.

In [ ]:
WEATHER_PREFIXES = {"temp", "prcp", "wspd", "wpgt", "pres", "snow", "coco"}
DELAY_CAUSES = {"CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"}

def column_group(col: str) -> str:
    if col.split("_")[0] in WEATHER_PREFIXES:
        return "weather (origin)" if col.endswith("_ORIGIN") else "weather (dest)"
    if col.startswith("DIV"):
        return "diversion details"
    if col in DELAY_CAUSES:
        return "delay causes"
    if col in {"YEAR", "QUARTER", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "FL_DATE"}:
        return "date"
    if col.startswith("OP_") or col == "TAIL_NUM":
        return "carrier"
    if col.startswith("ORIGIN"):
        return "origin"
    if col.startswith("DEST"):
        return "destination"
    if col in {"CRS_DEP_TIME", "TAXI_OUT", "WHEELS_OFF"} or col.startswith("DEP_"):
        return "departure"
    if col in {"CRS_ARR_TIME", "TAXI_IN", "WHEELS_ON"} or col.startswith("ARR_"):
        return "arrival"
    if col in {"CANCELLED", "CANCELLATION_CODE", "DIVERTED"}:
        return "cancellation / diversion"
    if col in {"FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME"}:
        return "gate return"
    return "flight summary"

columns = pd.DataFrame({
    "group": [column_group(c) for c in df.columns],
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(),
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
columns

### O4. Column groups

How many columns each group has, and how many of them are mostly empty (over 90% missing).

In [ ]:
groups = columns.groupby("group").agg(
    n_columns=("dtype", "size"),
    mostly_empty=("pct_missing", lambda s: (s > 90).sum()),
    avg_pct_missing=("pct_missing", "mean"),
).round(1).sort_values("n_columns", ascending=False)
groups

### O5. Sample rows

The first five rows, transposed so all columns are readable.

In [ ]:
df.head().T

### O6. Numeric summary

In [ ]:
df.describe().T

### O7. Categorical summary

Text columns: number of distinct values and the five most common.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

pd.DataFrame({
    "n_unique": [df[c].nunique() for c in text_cols],
    "top_5": [df[c].value_counts().head(5).to_dict() for c in text_cols],
}, index=text_cols)